<a href="https://colab.research.google.com/github/Rhuan-Messias/T-picosIA/blob/main/v4_ml_cat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import ResNet50, preprocess_input
from tensorflow.keras.preprocessing import image
import numpy as np
import pandas as pd
import os
import shutil
from scipy.spatial import distance
from sklearn.metrics import classification_report
import tensorflow_datasets as tfds
from PIL import Image

In [ ]:
# Carregar ResNet50 como extrator de características (sem a cabeça de classificação)
model_extractor = ResNet50(weights='imagenet', include_top=False, pooling='avg')
print("✅ Extrator ResNet50 configurado!")

In [ ]:
# Definir nomes dos 5 gatos para o exercício
nomes_gatos = ["Mimi", "Alfredo", "Bolinha", "Garfield", "Frajola"]

# Criar pastas de Treino e Teste
for gato in nomes_gatos:
    os.makedirs(f"treino/{gato}", exist_ok=True)
os.makedirs("teste", exist_ok=True)

# Baixar imagens para preencher a estrutura (110 imagens no total)
dataset_raw = tfds.load('cats_vs_dogs', split='train[:110]', as_supervised=True)
imgs = list(dataset_raw)

# Distribuir 20 fotos para treino de cada gato e 10 para o teste geral
print("📂 Organizando pastas de biometria...")
idx = 0
for gato in nomes_gatos:
    for i in range(20): # 20 fotos por gato [cite: 16]
        img_tensor, _ = imgs[idx]
        Image.fromarray(img_tensor.numpy()).save(f"treino/{gato}/foto_{i}.jpg")
        idx += 1

# 10 fotos para teste (gabarito) [cite: 17, 18]
gabarito_real = []
for i in range(10):
    gato_real = nomes_gatos[i % 5] # Distribui as 10 fotos entre os 5 gatos
    img_tensor, _ = imgs[idx]
    path_teste = f"teste/teste_{i}_gato_{gato_real}.jpg"
    Image.fromarray(img_tensor.numpy()).save(path_teste)
    gabarito_real.append({"path": path_teste, "label": gato_real})
    idx += 1

print("✅ Base de treino e teste pronta!")

In [ ]:
def get_embedding(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return model_extractor.predict(img_array, verbose=0).flatten()

centroides = {}

print("🧬 Calculando Centróides (Perfis Médios)...")
for gato in nomes_gatos:
    pasta = f"treino/{gato}"
    vetores = []
    for img_name in os.listdir(pasta):
        vetores.append(get_embedding(os.path.join(pasta, img_name)))
    # O centróide é a média dos vetores das 20 fotos [cite: 7, 21]
    centroides[gato] = np.mean(vetores, axis=0)

print("✅ Todos os 5 centróides foram calculados!")

In [ ]:
resultados_identificacao = []

print("🔎 Iniciando identificação das fotos de teste...")
for item in gabarito_real:
    embedding_teste = get_embedding(item['path'])

    # Comparar com cada centróide usando Distância Euclidiana
    distancias = {gato: distance.euclidean(embedding_teste, centroide)
                  for gato, centroide in centroides.items()}

    # O gato com a MENOR distância é o escolhido [cite: 22]
    predicao = min(distancias, key=distancias.get)

    resultados_identificacao.append({
        "Foto": os.path.basename(item['path']),
        "Rótulo_Real": item['label'],
        "Rótulo_Predito": predicao
    })

# Criar DataFrame com os resultados [cite: 23]
df_resultados = pd.DataFrame(resultados_identificacao)
display(df_resultados)

In [ ]:
print("\n📊 RELATÓRIO DE CLASSIFICAÇÃO (sklearn.metrics)")
report = classification_report(df_resultados["Rótulo_Real"],
                               df_resultados["Rótulo_Predito"],
                               target_names=nomes_gatos)
print(report)
